In [ ]:
##### IMPORTS

import os
os.environ['picaso_refdata'] = r'C:\Users\Alex\Desktop\Picaso\picaso\reference' # THIS MUST GO BEFORE YOUR IMPORT STATEMENT
os.environ['PYSYN_CDBS'] = r'C:\Users\Alex\Desktop\Picaso\grp\redcat\trds' # This is for the stellar data discussed below.

# General
import numpy as np
import astropy.units as u
import bd_support as sup

from pathlib import Path
from itertools import product

# Picaso and Virga
from picaso import justdoit as jdi
from virga import justdoit as vj

# To see what clouds are availible
# vj.available()

In [12]:
##### CONFIGURATIONS

# Directories
sonor_path  = r'C:\Users\Alex\Desktop\Picaso\data\sonora' # Sonora db
# sonor_path  = '/groups/tkaralidi/pbraunschweig/training_set/profiles/'
virga_path  = r'C:\Users\Alex\Desktop\Picaso\data\virga'  # Virga
# virga_path  = '/home/sa221179/picaso/virga/'
opaci_path  = None # Opacity db
# opcai_path  = '/groups/tkaralidi/opacity_500k_for_R5000_egpoutput.db'
output_path = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs")
# output_path = 'home/al864695/ouputs'

# Things that will remain constant
clouds      = ['Al2O3', 'CaTiO3', 'CH4', 'Cr', 'Fe',
               'H2O', 'KCl', 'Mg2SiO4', 'MgSiO3', 'MnS',
               'NH3', 'Na2S', 'TiO2', 'ZnS']

# Constant values
wav_range   = [0.3, 5.0] # microns
MH          = 1.0        # [M/H] metallicity factor ~ solar
MU          = 2.36       # Average MU
R           = 300        # resolution
# R           = 5000

In [13]:
##### METHOD BROWN DWARF SPECTRUM

def bd_spectrum(Teff, logg, fsed, kzz):
    """
    Compute a BD emission spectrum with Virga clouds.
    """

    # Opacity & inputs
    opa    = jdi.opannection(wav_range, opaci_path)
    bd     = jdi.inputs(calculation="browndwarf")
    bd.phase_angle(0)
    # Convert log g [cgs] → grav [m s^-2]
    gravity = 10**logg * 1e-2   # 1 cm s^-2 = 0.01 m s^-2
    bd.gravity(gravity, gravity_unit=u.Unit('m/s**2'))
    bd.sonora(sonor_path, Teff)

    # Inject corrected TP
    sup.inject_corr(bd, Teff, logg, fsed)

    # Inject Kzz (match pressure grid length)
    prof   = bd.inputs['atmosphere']['profile']
    P      = np.asarray(prof["pressure"], float)
    bd.inputs["atmosphere"]["profile"]["kz"] = [float(kzz)] * len(P)

    # Clouds
    bd.virga(clouds, virga_path, fsed, mh=MH, mmw=MU)
    out    = bd.spectrum(opa, full_output=True)

    wn, fl = out["wavenumber"], out["thermal"]  # cm^-1 and erg cm^-2 s^-1 cm^-1
    wn, fl = jdi.mean_regrid(wn, fl, R=R)

    # Convert wavenumber [cm^-1] to wavelength [micron]
    w_um = 1e4 / wn
    # Flux
    #F_R_um = F_R * (1e4 / (w_um**2))

    # Sort to ascending wavelength
    #idx    = np.argsort(w_um)
    #w_um, flux = w_um[idx], F_R_um[idx]

    # Save to output dictionary
    #out['regridx']  = w_um
    #out['regridy']  = flux

    return w_um, fl

In [14]:
##### GENERATE AND SAVE SPECTRUM

# Sample test, but can replace with correct PARAMETER SPACE
Teff_s = [1450]      # K
logg_s = [4.5]       # log g cgs
fsed_s = [1.0, 2.0] 
kzz_s  = [1e9, 1e10]
  
for Teff_i, logg_i, fsed_i, kzz_i in product(Teff_s, logg_s, fsed_s, kzz_s):

    # Run spectrum
    W_i, F_i = bd_spectrum(Teff_i, logg_i, fsed_i, kzz_i)

    # Short, readable filename: t<T>g<g>f<fsed><kTag>.npz
    fname = f"t{int(Teff_i)}g{int(logg_i)}f{int(fsed_i)}{sup.format_kzz(kzz_i)}.npz"
    fpath = output_path / fname

    # Save
    np.savez_compressed(fpath,
                        x=np.array([Teff_i, logg_i, fsed_i, kzz_i], dtype=float),
                        y=F_i.astype(float),
                        wavelength_um=W_i.astype(float))

print(f"Saved {len(Teff_s)*len(logg_s)*len(fsed_s)*len(kzz_s)} files to {output_path}")

Saved 4 files to C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs
